# Exploração da API AISWEB (DECEA)

O AISWEB é o portal de informação aeronáutica do DECEA. A API dele fica em `https://aisweb.decea.mil.br/api/`, autentica por query string (`apiKey` + `apiPass`) e responde **sempre em XML**. Cada "rota" é escolhida pelo parâmetro `area`.

As credenciais ficam no arquivo `.env` (que está no `.gitignore`), nunca no notebook.

### Rotas mapeadas (testadas em 05/10/2026)

| `area` | O que retorna | Parâmetros que funcionam |
|---|---|---|
| `localidades` | Todos os indicadores ICAO (~6,1 mil) com o tipo | — |
| `rotaer` | **Sem** `icaoCode`: lista paginada de aeródromos. **Com** `icaoCode`: ficha completa (pistas, frequências, auxílios, observações) | `icaoCode`, `uf`, `city`, `name`, `type`, `rowstart` |
| `met` | METAR e TAF mais recentes | `icaoCode` (um por chamada) |
| `sol` | Nascer e pôr do sol (UTC) | `icaoCode`, `dt_i`, `dt_f` |
| `notam` | NOTAMs vigentes | `icaocode` (sem ele, retorna todos, ~1,8 mil) |
| `cartas` | Cartas aeronáuticas (SID, STAR, IAC, ADC, VAC...) com link do PDF | `icaoCode`, `tipo` |
| `suplementos` | Suplementos AIP (AIP SUP) | — |
| `infotemp` | Informações temporárias de aeródromo | `icaoCode` |
| `pub` | Publicações da AIP (emendas, AIP completa, ENRC) | `type=AIP` (único valor aceito) |
| `waypoints` | Fixos/waypoints com lat/lon (~6,8 mil) | `rowstart` |

### Pegadinhas descobertas
- **Paginação**: `rotaer` (lista) e `waypoints` devolvem 100 itens por vez. `rowstart` é o *offset* (base 0): `rowstart=0`, `100`, `200`...
- **`met` com vários ICAOs** separados por vírgula devolve só o primeiro. Faça uma chamada por aeródromo.
- **`notam` com vários ICAOs** separados por vírgula ignora o filtro e devolve tudo. Baixe tudo e filtre no pandas.
- **`pub` com `type` diferente de `AIP`** quebra no servidor e o erro dispara um e-mail automático para a equipe do DECEA. Não fique testando valores ali.
- Os links de download das `cartas` **contêm a sua `apiKey`**. O notebook esconde essa coluna na exibição. Limpe os outputs antes de commitar.

## Setup

In [1]:
import os
import time
import xml.etree.ElementTree as ET
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests

from module_decea.config import EXTERNAL_DATA_DIR  # o import também carrega o .env da raiz

API_KEY = os.environ["AISWEB_API_KEY"]
API_PASS = os.environ["AISWEB_API_PASS"]
BASE_URL = "https://aisweb.decea.mil.br/api/"

SESSION = requests.Session()

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

## Funções auxiliares

- `aisweb(area, **params)`: faz a chamada e devolve a raiz do XML. Levanta `AiswebErro` quando a API responde com erro.
- `itens(root)`: pega os `<item>` de dentro do contêiner da resposta (`<notam>`, `<cartas>`, `<rotaer>`...).
- `para_df(elementos)`: transforma uma lista de elementos XML em DataFrame, usando os filhos "folha" (sem sub-elementos) como colunas.
- `listar_paginado(area, **params)`: percorre as rotas paginadas com `rowstart`.

In [2]:
class AiswebErro(Exception):
    pass


def aisweb(area, **params):
    query = {"apiKey": API_KEY, "apiPass": API_PASS, "area": area, **params}
    resp = SESSION.get(BASE_URL, params=query, timeout=60)
    resp.raise_for_status()
    try:
        root = ET.fromstring(resp.content)
    except ET.ParseError:
        # Erros internos do servidor voltam como HTML com status 200
        raise AiswebErro(f"Resposta não-XML para area={area!r}: {resp.text.strip()[:200]!r}")

    # A API sinaliza erro de dois jeitos: <x total="error"><msg>...</msg></x> ou <error>...</error>
    erro = root.find(".//*[@total='error']")
    if erro is not None:
        raise AiswebErro(erro.findtext("msg", "").strip())
    if root.find("error") is not None:
        raise AiswebErro(root.findtext("error").strip())
    return root


def itens(root):
    return root.findall("./*/item")


def total(root):
    # Alguns totais vêm com espaços em volta (ex.: total="  62  ")
    return int(root[0].get("total", "0").strip())


def para_df(elementos):
    linhas = []
    for el in elementos:
        linhas.append({filho.tag: (filho.text or "").strip() for filho in el if len(filho) == 0})
    return pd.DataFrame(linhas)


def listar_paginado(area, max_paginas=None, pausa=0.2, **params):
    elementos, rowstart, paginas = [], 0, 0
    while True:
        root = aisweb(area, rowstart=rowstart, **params)
        pagina = itens(root)
        elementos.extend(pagina)
        paginas += 1
        rowstart += len(pagina)
        if not pagina or rowstart >= total(root) or paginas == max_paginas:
            break
        time.sleep(pausa)
    print(f"{area}: {len(elementos)} itens em {paginas} página(s), total informado = {total(root)}")
    return elementos

## 1. `localidades`: todos os indicadores ICAO

Tipos encontrados: `AD` aeródromo, `HP` heliponto, `HD` (9 registros, não documentado), `NOT` indicadores usados só em NOTAM (órgãos, ex.: `SBEA` = DIRAD).

In [3]:
df_loc = para_df(itens(aisweb("localidades")))
print(len(df_loc), "localidades")
df_loc["type"].value_counts()

6145 localidades


type
AD     4998
HP     1090
NOT      48
HD        9
Name: count, dtype: int64

## 2. `rotaer` (modo lista): busca de aeródromos

Sem `icaoCode`, a rota vira uma busca. Filtros que funcionam: `uf`, `city`, `name`, `type` (`AD`, `HP`...). Cada resposta traz no máximo 100 itens; o atributo `total` diz quantos existem.

In [4]:
root = aisweb("rotaer", uf="SP")
print("atributos do contêiner:", root[0].attrib)
para_df(itens(root)).head(10)

atributos do contêiner: {'pagesize': '100', 'total': '953'}


,id,type,AeroCode,ciad,name,city,uf,lng,lat,dt
0,0822c7d2-700f-4977-a59f-596542aca57b,AD,SJ18,SP1521,Fazenda Vista Alegre,Sud Mennucci,SP,-50.843611111111,-20.661388888889,2026-10-01
1,92ac9bd2-a025-4dc5-9825-82c7f0dcf8aa,HP,SJ31,SP1531,ML3 Empreendimentos,Guararema,SP,-46.011111111111,-23.421666666667,2026-10-01
2,1fe663b6-4f0e-43db-aef7-aebc1b8dce47,AD,SBSR,SP0006,Professor Eribelto Manoel Reino,São José do Rio Preto,SP,-49.406944444444,-20.817222222222,2026-09-25
3,f94f759d-d4c6-4f30-bd53-7025cff51586,AD,SBJD,SP0031,Comandante Rolim Adolfo Amaro,Jundiaí,SP,-46.943611111111,-23.181666666667,2026-09-24
4,a25462dd-91b9-465c-85b1-f9c51741eef7,AD,SBTA,SP9004,Base de Aviação de Taubaté,Taubaté,SP,-45.515833333333,-23.038888888889,2026-09-24
5,1efbc9d7-77b9-47c0-b6ef-74bb6aa9832d,HP,SDJD,SP0360,Juquehy Baleia,São Sebastião,SP,-45.7125,-23.760833333333,2026-09-24
6,0a084070-17f6-4014-85dd-32b1ede8d5a3,HP,SJ30,SP1530,DAS Brasil,Taubaté,SP,-45.510555555556,-23.033333333333,2026-09-24
7,706252bd-9545-4f90-a7fd-a6a1063da44e,HP,SNN2,SP0784,Helicenter Castilho,Arujá,SP,-46.330555555556,-23.408888888889,2026-09-24
8,353dafcb-1ee6-43d4-b997-9b62256b3696,AD,SDAA,SP0045,Araras,Araras,SP,-47.358333333333,-22.336944444444,2026-09-17
9,ac95659e-acb8-4944-a908-ce4849a52752,AD,SI70,SP1528,Estância Ipanema,São José do Rio Preto,SP,-49.353055555556,-20.9175,2026-09-17


In [5]:
# Outros filtros
display(para_df(itens(aisweb("rotaer", city="Campinas"))).head())
display(para_df(itens(aisweb("rotaer", name="Viracopos"))))

,id,type,AeroCode,ciad,name,city,uf,lng,lat,dt
0,49c845a0-78e4-4af9-8953-25d466856d7a,HP,SIA1,SP1523,Shopping Parque Dom Pedro,Campinas,SP,-47.065833333333,-22.845,2026-08-13
1,4c3c6d2d-cdd0-4406-a93d-a4c51457a53d,AD,SBKP,SP0003,Viracopos,Campinas,SP,-47.134444444444,-23.006944444444,2026-07-30
2,e39bb1af-f73f-41eb-8967-00b64dc43047,AD,SDAM,SP0037,Campos dos Amarais - Prefeito Francisco Amaral,Campinas,SP,-47.108055555556,-22.859166666667,2026-07-09
3,83b2f278-d26d-47a4-be92-721a9255c1a5,HP,SISB,SP0549,Basalto Base 6,Campinas,SP,-47.088333333333,-23.003888888889,2026-03-05
4,f006d297-8f2e-4e99-a1d6-7a1e6d186434,HP,SDFB,SP0325,Fundação Bradesco,Campinas,SP,-47.077222222222,-22.964444444444,2025-10-30


,id,type,AeroCode,ciad,name,city,uf,lng,lat,dt
0,4c3c6d2d-cdd0-4406-a93d-a4c51457a53d,AD,SBKP,SP0003,Viracopos,Campinas,SP,-47.134444444444,-23.006944444444,2026-07-30


In [6]:
# Lista completa (com paginação) de aeródromos de SP
df_ad_sp = para_df(listar_paginado("rotaer", uf="SP", type="AD"))
df_ad_sp.head()

rotaer: 320 itens em 4 página(s), total informado = 320


,id,type,AeroCode,ciad,name,city,uf,lng,lat,dt
0,0822c7d2-700f-4977-a59f-596542aca57b,AD,SJ18,SP1521,Fazenda Vista Alegre,Sud Mennucci,SP,-50.843611111111,-20.661388888889,2026-10-01
1,1fe663b6-4f0e-43db-aef7-aebc1b8dce47,AD,SBSR,SP0006,Professor Eribelto Manoel Reino,São José do Rio Preto,SP,-49.406944444444,-20.817222222222,2026-09-25
2,f94f759d-d4c6-4f30-bd53-7025cff51586,AD,SBJD,SP0031,Comandante Rolim Adolfo Amaro,Jundiaí,SP,-46.943611111111,-23.181666666667,2026-09-24
3,a25462dd-91b9-465c-85b1-f9c51741eef7,AD,SBTA,SP9004,Base de Aviação de Taubaté,Taubaté,SP,-45.515833333333,-23.038888888889,2026-09-24
4,353dafcb-1ee6-43d4-b997-9b62256b3696,AD,SDAA,SP0045,Araras,Araras,SP,-47.358333333333,-22.336944444444,2026-09-17


## 3. `rotaer` (modo ficha): detalhes de um aeródromo

Com `icaoCode`, a resposta é aninhada: pistas, serviços (COM, NAV, MET, AIS...), distâncias declaradas e observações. Troque o `ICAO` abaixo para explorar outro aeródromo.

In [7]:
ICAO = "SBGR"
ficha = aisweb("rotaer", icaoCode=ICAO)

campos = ["AeroCode", "name", "city", "uf", "lat", "lng", "altFt", "type", "typeUtil",
          "typeOpr", "cat", "utc", "fir", "jur", "status", "dt"]
geral = {c: ficha.findtext(c) for c in campos}
geral["operador"] = ficha.findtext("org/name")
pd.Series(geral)

AeroCode                                           SBGR
name        Guarulhos - Governador André Franco Montoro
city                                          São Paulo
uf                                                   SP
lat                                    -23.435555555556
lng                                    -46.473055555556
altFt                                              2461
type                                                 AD
typeUtil                                        PUB/MIL
typeOpr                                         VFR IFR
cat                                                INTL
utc                                                  -3
fir                                                SBBS
jur                                            CRCEA-SE
status                                           Active
dt                                           2026-06-25
operador                                    GRU Airport
dtype: str

In [8]:
# Pistas
pistas = pd.DataFrame([
    {
        "pista": rwy.findtext("ident"),
        "superficie": rwy.findtext("surface"),
        "comprimento_m": rwy.findtext("length"),
        "largura_m": rwy.findtext("width"),
        "resistencia_pcn": rwy.findtext("surface_c"),
        "cabeceiras": ", ".join(t.findtext("ident") for t in rwy.findall("thr")),
    }
    for rwy in ficha.findall("runways/runway")
])
pistas

,pista,superficie,comprimento_m,largura_m,resistencia_pcn,cabeceiras
0,10R/28L,ASPH,3000,45,810/F/C/X/T,"10R, 28L"
1,10L/28R,ASPH,3700,45,790/F/C/X/T,"10L, 28R"


In [9]:
# Distâncias declaradas por cabeceira
para_df(ficha.findall("rmkDistDeclared/rmkDist"))

,rwy,tora,toda,asda,lda,aux,altGeoidal,lat,lng
0,10L,3700,3700,3760,3610,,"- 2,33",S 23 26 03,W 046 28 57
1,28R,3700,3700,3760,3640,,"- 2,35",S 23 25 30,W 046 26 57
2,10R,3000,3300,3060,3000,,"- 2,33",S 23 26 20,W 046 29 13
3,28L,3000,3000,3060,3000,,"- 2,35",S 23 25 52,W 046 27 32


In [10]:
# Frequências de comunicação
freqs = pd.DataFrame([
    {
        "orgao": svc.findtext("type"),
        "indicativo": svc.findtext("callsign"),
        "frequencias": ", ".join(f.text for f in svc.findall("freqs/freq")),
    }
    for svc in ficha.findall("services/service[@type='COM']")
])
freqs

,orgao,indicativo,frequencias
0,Torre,Guarulhos,"118.400, 121.500, 132.750, 135.200"
1,Solo,Guarulhos,"121.700, 126.900"
2,Operações,Guarulhos,122.500
3,Tráfego,Guarulhos,121.000
4,ATIS,São Paulo,127.750


In [11]:
# Auxílios à navegação (ILS, VOR, NDB...)
para_df(ficha.findall("services/service[@type='NAV']"))

,type,cat,thr,ident,freq,lat,lng
0,ILS/DME,,28L,IBC,111.1,2326.38S,04629.41W
1,ILS/DME,,10R,IGH,111.7,2325.82S,04627.33W
2,ILS/DME,,28R,IGS,111.9,2326.13S,04629.29W
3,ILS/DME,,10L,IUC,110.7,2325.41S,04626.60W
4,IM,,10L,IUC,75,2326.10S,04629.17W


In [12]:
# Observações (o atributo 'cod' é a seção da AIP)
obs = pd.DataFrame([{"secao": r.get("cod"), "texto": (r.text or "").strip()} for r in ficha.findall("rmk/rmkText")])
obs.head(15)

,secao,texto
0,2.6,"Plano de Remoção de ACFT inoperantes (PRAI): Capacidade para remoção até ACFT A380-800 - Peso 575.000kg, sob respons..."
1,2.8,PRKG de ACFT da aviação civil geral somente no pátio NR 12 e mediante AUTH do centro de controle operacional através...
2,2.8,Pátio 12 PRKG HEL com ABVL de 3 posições.
3,2.8,"TWY SIERRA, TANGO e UNIFORME CLSD para ACFT dimensões\nABV B-707, TAX por meios próprios."
4,2.8,AUTH voos de carga e voos não regulares somente 0300/0900 (com\ncapacidadede posição PRKG para 12 PSN) e 1500/2100 (...
5,2.8,AUTH voos de rede postal noturna somente 0230/0900\ncompermanência MAX no solo de 2 HR.
6,2.8,"Restrição de longa permanência mais de 3 HR para voos\nextras,translados e não regulares de passageiros e cargas."
7,2.8,TWY Alfa BTN TWY Golf e pátio 1 LTD para ACFT envergadura\nmáxima 65m.
8,2.8,TWY Victor para ACFT envergadura máxima 65M.
9,2.8,ACFT MIL destino BASP (pátio 13) deverão CTC OPS Guarulhos (FREQ 122.50) de 0800-0000 fim COOR apoio GND. Apoio fora...


## 4. `met`: METAR e TAF

Uma chamada por aeródromo. Aeródromos sem estação meteorológica voltam com `<metar/>` vazio.

In [13]:
def metar_taf(icao):
    met = aisweb("met", icaoCode=icao).find("met")
    return {"icao": icao, "metar": met.findtext("metar") or None, "taf": met.findtext("taf") or None}


df_met = pd.DataFrame([metar_taf(i) for i in ["SBGR", "SBSP", "SBKP", "SBRJ", "SBGL", "SBBR", "SDJV"]])
df_met

,icao,metar,taf
0,SBGR,METAR SBGR 061000Z 06007KT 1200 R10R/P2000 R10L/P2000 R28R/P2000 R28L/P2000 BR OVC002 17/17 Q1020=,TAF SBGR 060430Z 0606/0712 10005KT 8000 BKN007 OVC011 TN17/0608Z TX23/0618Z TEMPO 0606/0611 4000 DZ BR BECMG 0611/06...
1,SBSP,METAR SBSP 061000Z 09005KT 9999 OVC006 18/17 Q1021=,TAF SBSP 060430Z 0606/0618 10005KT 9999 BKN009 TN17/0607Z TX22/0615Z PROB30 TEMPO 0607/0611 6000 RA BECMG 0611/0613 ...
2,SBKP,METAR SBKP 061000Z 11005KT CAVOK 19/17 Q1018=,TAF SBKP 060430Z 0606/0706 10006KT 5000 BR NSC TN18/0608Z TX24/0615Z PROB30 0609/0611 02005KT 3000 DZ BR BKN003 BECM...
3,SBRJ,METAR SBRJ 061000Z 02003KT 9999 FEW008 SCT012 23/21 Q1017=,TAF SBRJ 060425Z 0606/0618 33005KT 7000 SCT020 BKN035 TN22/0609Z TX25/0617Z TEMPO 0606/0612 07007KT 3500 DZ BR BKN01...
4,SBGL,METAR SBGL 061000Z 28003KT 9999 BKN010 BKN100 22/21 Q1018=,TAF SBGL 060425Z 0606/0712 31005KT 7000 SCT020 BKN035 TN21/0609Z TX27/0617Z TEMPO 0606/0612 06007KT 3500 DZ BR BKN00...
5,SBBR,METAR SBBR 061000Z VRB02KT 7000 -RA BKN010 OVC020 19/19 Q1021 RERA=,TAF SBBR 060400Z 0606/0706 12003KT 9999 BKN020 TN18/0609Z TX29/0618Z PROB30 0606/0611 6000 BKN006 BECMG 0612/0614 05...
6,SDJV,NaN,NaN


## 5. `sol`: nascer e pôr do sol

Horários em **UTC** (Brasília = UTC−3). `dt_i`/`dt_f` definem o intervalo de datas.

In [14]:
hoje = date.today()
root = aisweb("sol", icaoCode="SBGR", dt_i=hoje.isoformat(), dt_f=(hoje + timedelta(days=6)).isoformat())
para_df(root.findall("day"))

,date,sunrise,sunset,weekDay,aero,geo
0,2026-10-06,08:40,21:07,2,SBGR,"-23.435555555556,-46.473055555556"
1,2026-10-07,08:39,21:07,3,SBGR,"-23.435555555556,-46.473055555556"
2,2026-10-08,08:38,21:07,4,SBGR,"-23.435555555556,-46.473055555556"
3,2026-10-09,08:37,21:08,5,SBGR,"-23.435555555556,-46.473055555556"
4,2026-10-10,08:36,21:08,6,SBGR,"-23.435555555556,-46.473055555556"
5,2026-10-11,08:35,21:09,7,SBGR,"-23.435555555556,-46.473055555556"
6,2026-10-12,08:34,21:09,1,SBGR,"-23.435555555556,-46.473055555556"


## 6. `notam`

Campos principais (seguem o formato ICAO de NOTAM):

| campo | significado |
|---|---|
| `n` | número do NOTAM (série + número/ano) |
| `tp` | `NOTAMN` novo, `NOTAMR` substitui (`ref`), `NOTAMC` cancela |
| `cod` | código Q (ex.: `QOBCE` = obstáculo erguido) |
| `b` / `c` | início e fim da vigência (`AAMMDDhhmm` UTC; `c` pode ser `PERM`) |
| `d` | período de atividade dentro da vigência |
| `e` | texto do NOTAM |
| `f` / `g` | limites inferior e superior |
| `geo` | coordenada e raio da área afetada |

In [15]:
df_notam = para_df(itens(aisweb("notam", icaocode="SBGR")))
df_notam["inicio"] = pd.to_datetime(df_notam["b"], format="%y%m%d%H%M", errors="coerce")
df_notam["fim"] = pd.to_datetime(df_notam["c"], format="%y%m%d%H%M", errors="coerce")
df_notam[["n", "tp", "cod", "inicio", "fim", "e"]]

,n,tp,cod,inicio,fim,e
0,F4819/26,NOTAMR,QOBCE,2026-08-26 21:12:00,2026-11-24 03:00:00,"OBST MONTADO (GUINDASTE) LGTD COORD 232537.96S/0462446.88W, HGT 35M (115FT), ALT OTP 796,00M (2612FT) RAIO 30M"
1,F5139/26,NOTAMN,QOBCE,2026-09-03 15:18:00,2026-12-02 23:59:00,"OBST (GUINDASTE) MONTADO LGTD COORD 232540.24S/0462502.75W HGT 21,0 M (69FT) ALT OTP 817,0 M (2680FT) RAIO 20M"
2,F5317/26,NOTAMR,QOBCE,2026-09-08 22:46:00,2026-12-07 03:00:00,OBST MONTADO (GUINDASTE) LGTD COORD 232515.01S0462706.69W HGT 45M (148FT) ALT OTP 787.68M (2584FT) RAIO 19.09M
3,F5954/26,NOTAMN,QOBCE,2026-10-01 13:39:00,2026-12-29 03:00:00,OBST (GUINDASTE) LGTD MONTADO WI COORD 232540.40S0462836.30W - 232540.80S0462837.90W - 232535.00S0462839.40W - 2325...
4,F6009/26,NOTAMR,QMRLC,2026-10-06 05:00:00,2026-10-27 07:59:00,RWY 10L/28R CLSD DEVIDO SER MAINT
5,F6014/26,NOTAMN,QMRLC,2026-10-07 03:30:00,2026-10-31 07:59:00,RWY 10R/28L CLSD DEVIDO SER MAINT


In [16]:
# Todos os NOTAMs vigentes do Brasil (~4 MB de XML)
df_notam_all = para_df(itens(aisweb("notam")))
print(len(df_notam_all), "NOTAMs")
df_notam_all["icaoairport_id"].value_counts().head(15)

1774 NOTAMs


icaoairport_id
        406
SBRJ     43
SBSP     33
SBRF     19
SBBR     17
SBGL     16
SBGO     16
SBCF     14
SBCO     14
SBFZ     14
SBMT     13
SBSJ     12
SBSV     11
SBVT     11
SBTA     10
Name: count, dtype: int64

In [17]:
# Por tipo e por assunto (letras 2 e 3 do código Q)
display(df_notam_all["tp"].value_counts())
df_notam_all["cod"].str[1:3].value_counts().head(15)

tp
NOTAMN    1191
NOTAMR     583
Name: count, dtype: int64

cod
RT    492
WP    140
OB    116
XX    115
WU     97
FA     92
RD     77
RR     67
MR     61
WL     49
WG     44
PI     38
MX     35
NM     33
WE     31
Name: count, dtype: int64

## 7. `cartas`: cartas aeronáuticas

Filtros: `icaoCode` e `tipo` (`SID`, `STAR`, `IAC`, `ADC`, `VAC`, `PDC`, `AOC`, `ARC`...). A coluna `link` traz a sua `apiKey`, então ela fica fora das exibições.

In [18]:
COLS_CARTAS = ["IcaoCode", "tipo", "tipo_descr", "nome", "dt", "amdt", "especie"]

df_cartas = para_df(itens(aisweb("cartas", icaoCode="SBGR")))
display(df_cartas["tipo"].value_counts())
df_cartas[COLS_CARTAS].head(10)

tipo
IAC     24
SID     15
STAR     9
AGMC     5
AOC      4
PDC      3
ADC      1
VAC      1
Name: count, dtype: int64

,IcaoCode,tipo,tipo_descr,nome,dt,amdt,especie
0,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV REORI 3A RWY 10R,2026-10-01,2610A1,Convencionais
1,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV EVNEB 3A RWY 28L,2026-10-01,2610A1,Convencionais
2,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV CGO 5A RWY 10R,2026-10-01,2610A1,Convencionais
3,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV EKOPO 1A RWY 10L,2026-10-01,2610A1,Convencionais
4,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV ORONU 1A RWY 10L / 10R,2026-10-01,2610A1,Convencionais
5,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV ZORZA 3A RWY 10L,2026-10-01,2610A1,Convencionais
6,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV AMVUL 6A RWY 10L,2026-10-01,2610A1,Convencionais
7,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV NIBRU 3A - UREMI 3A RWY 10R,2026-10-01,2610A1,Convencionais
8,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV ORONU 1B RWY 10R/10L,2026-10-01,2610A1,Convencionais
9,SBGR,SID,Carta de Saída Padrão por Instrumentos,RNAV CGO 4B RWY 28L,2026-10-01,2610A1,Convencionais


In [19]:
# Só as cartas de aproximação por instrumentos (IAC) de Guarulhos
para_df(itens(aisweb("cartas", icaoCode="SBGR", tipo="IAC")))[COLS_CARTAS]

,IcaoCode,tipo,tipo_descr,nome,dt,amdt,especie
0,SBGR,IAC,Carta de aproximação por instrumentos,RNP M RWY 28R,2026-03-19,2603A1,Convencionais
1,SBGR,IAC,Carta de aproximação por instrumentos,ILS W CAT I & II RWY 10L,2026-03-19,2603A1,Convencionais
2,SBGR,IAC,Carta de aproximação por instrumentos,RNP W RWY 10L,2026-03-19,2603A1,Convencionais
3,SBGR,IAC,Carta de aproximação por instrumentos,ILS M RWY 28R,2026-03-19,2603A1,Convencionais
4,SBGR,IAC,Carta de aproximação por instrumentos,ILS R RWY 10R,2025-04-17,2504A1,Convencionais
5,SBGR,IAC,Carta de aproximação por instrumentos,RNP S RWY 10L (AR),2024-01-25,2401A1,Convencionais
6,SBGR,IAC,Carta de aproximação por instrumentos,ILS Y RWY 10R,2022-11-03,2211A1,Convencionais
7,SBGR,IAC,Carta de aproximação por instrumentos,ILS Z OR LOC Z RWY 28L,2022-09-08,2209A1,Convencionais
8,SBGR,IAC,Carta de aproximação por instrumentos,ILS T RWY 28R,2022-09-08,2209A1,Convencionais
9,SBGR,IAC,Carta de aproximação por instrumentos,ILS P CAT II & III RWY 10R,2022-09-08,2209A1,Convencionais


In [20]:
# Todas as cartas do Brasil (~2 MB de XML)
df_cartas_all = para_df(itens(aisweb("cartas")))
print(len(df_cartas_all), "cartas")
display(df_cartas_all["tipo"].value_counts())
df_cartas_all["IcaoCode"].value_counts().head(10)

1807 cartas


tipo
IAC        687
SID        488
ADC        136
STAR       133
VAC        124
PDC        114
AOC         51
AGMC        28
ATCSMAC     26
ARC         20
Name: count, dtype: int64

IcaoCode
SBBR    82
SBGR    62
SBGL    43
SBCT    40
SBCF    36
SBRJ    35
SBVT    34
SBRF    31
SBSP    29
SBKP    29
Name: count, dtype: int64

In [21]:
# Baixar o PDF de uma carta para data/external/cartas (pasta ignorada pelo git)
def baixar_carta(carta, pasta=EXTERNAL_DATA_DIR / "cartas"):
    destino = Path(pasta) / f"{carta['arquivo']}.pdf"
    destino.parent.mkdir(parents=True, exist_ok=True)
    resp = SESSION.get(carta["link"], timeout=120)
    resp.raise_for_status()
    destino.write_bytes(resp.content)
    return destino


carta = df_cartas[df_cartas["tipo"] == "ADC"].iloc[0]
baixar_carta(carta)

PosixPath('downloads/sbgr_sbgr-adc_adc_20260514.pdf')

## 8. `suplementos`: AIP SUP

In [22]:
df_sup = para_df(itens(aisweb("suplementos")))
print(len(df_sup), "suplementos")
display(df_sup["status"].value_counts())
df_sup[["n", "serie", "tipo", "local", "titulo", "status", "duracao"]].head(10)

86 suplementos


status
em vigor    86
Name: count, dtype: int64

,n,serie,tipo,local,titulo,status,duracao
0,97,N,airac,SBGV,AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 2300 UTC até 31 de Dezembro de 2026, 0959 UTC."
1,99,N,airac,SBBU,APROXIMAÇÃO E ILUMINAÇÃO DE PISTA,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 31 de Março de 2027, 2359 UTC."
2,100,N,airac,SBCF,"DADOS DE PÁTIOS, PISTAS DE TÁXI E PONTOS DE VERIFICAÇÃO",em vigor,"De 29 de Outubro de 2026, 0000 UTC até 28 de Agosto de 2027, 0300 UTC."
3,101,N,airac,SWJP,CARTAS RELACIONADAS AO AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 28 de Agosto de 2027, 2359 UTC."
4,102,N,airac,SWBV,CARTAS RELACIONADAS AO AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 28 de Agosto de 2027, 2359 UTC."
5,103,N,airac,SWKU,CARTAS RELACIONADAS AO AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 28 de Agosto de 2027, 2359 UTC."
6,104,N,airac,SBSP,REGULAMENTOS LOCAIS DE AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 20 de Agosto de 2029, 2359 UTC."
7,105,N,airac,SWPC,CARTAS RELACIONADAS AO AERÓDROMO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 28 de Agosto de 2027, 2359 UTC."
8,106,N,airac,SBPG,ATS INSTALAÇÕES DE COMUNICAÇÃO,em vigor,"De 29 de Outubro de 2026, 0000 UTC até 11 de Dezembro de 2026, 2359 UTC."
9,89,N,airac,SBPG,PROCEDIMENTO DE APROXIMAÇÃO POR INSTRUMENTO,em vigor,"De 01 de Outubro de 2026, 0000 UTC até 07 de Julho de 2027, 2359 UTC."


## 9. `infotemp`: informações temporárias de aeródromo

In [23]:
df_infotemp = para_df(itens(aisweb("infotemp")))
df_infotemp[["AeroCode", "number", "action", "aipsection", "startdate", "enddate", "rmk"]]

,AeroCode,number,action,aipsection,startdate,enddate,rmk
0,SBES,SP0954R/23,Restrição,REGULAMENTOS LOCAIS DE AERÓDROMO,{ts '2023-03-27 00:00:00'},{ts '2359-06-24 23:59:00'},EXER MIL SUBJ AUTH/COOR APP-ES ACONTECERA WI COORD 22 37 40S/042 15 53W; 22 45 08S/042 14 01W; 22 40 59S/042 10 01W;...
1,SBTA,SP0181M/21,Modificação,INFORMAÇÕES METEOROLÓGICAS FORNECIDAS,{ts '2021-02-02 19:53:03'},{ts '2300-03-29 00:00:00'},MET CMA AUTOATENDIMENTO HR SER MODIFICADO PARA FEB 02 TIL 14 17 TIL MAR 29 MON TIL FRI 0900-0300 SAT SUN HOL 0900-SS...
2,SSKW,AZ1747R/20,Restrição,HORÁRIO DE FUNCIONAMENTO,{ts '2020-10-27 20:54:22'},{ts '2028-02-21 22:00:00'},AIS HR SER DLY 1000-2200 INSTL
3,SSKW,AZ1742M/20,Modificação,AUXÍLIOS-RÁDIO A NAVEGAÇÃO E POUSO,{ts '2020-10-27 17:19:58'},{ts '2028-02-21 22:00:00'},AFIS CACOAL FREQ 131.100MHZ HR SER DLY 1000-2200 INSTL
4,SSKW,AZ1741M/20,Modificação,INFORMAÇÕES METEOROLÓGICAS FORNECIDAS,{ts '2020-10-27 17:10:36'},{ts '2028-02-21 22:00:00'},MET CMA (1 A 3) HR SER DLY 1000-2200 INSTL
5,SDTY,BS1691R/20,Restrição,"OUTRAS ILUMINAÇÕES, FONTE DE ENERGIA SECUNDÁRIA",{ts '2020-10-14 16:29:24'},{ts '2121-01-12 03:00:00'},ABN U/S
6,SSDK,BS0463R/19,Restrição,PROCEDIMENTOS DE VOO,{ts '2019-07-19 12:35:44'},{ts '2109-07-24 23:59:00'},CIRCUITO DE TFC ACFT ASA FIXA E HEL DEVERA SER REALIZADO HGT MNM 1100FT SOBRE ELEV AD


## 10. `pub`: publicações da AIP

In [24]:
para_df(itens(aisweb("pub", type="AIP")))

,id,type,name,file,amdt,amdt_number
0,2318,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=586c2260-b753-40ab-a96aa271a09fcf9b.pdf&p=AMDT"">AMDT",{ts '2026-09-03 00:00:00'},AMDT 36-26
1,2318,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=5be4cfb4-885e-490c-a8ddda9b51caa095.pdf&p=AMDT"">AMDT",{ts '2026-09-03 00:00:00'},AMDT 36-26
2,2318,AIP,Completa,"https://aisweb.decea.gov.br/download/?public=94038a83-e55d-41e2-bf197639e0cf34c4.pdf&p=Completa"">Completa",{ts '2026-09-03 00:00:00'},AMDT 36-26
3,2318,AIP,ENRC,"https://aisweb.decea.gov.br/download/?public=77a89b77-97ff-4ec4-a838a2e5e9656265.zip&p=ENRC"">ENRC",{ts '2026-09-03 00:00:00'},AMDT 36-26
4,2322,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=b7afd0d2-0105-4749-87b89de0b063cb86.pdf&p=AMDT"">AMDT",{ts '2026-10-01 00:00:00'},AMDT 40-26
5,2322,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=6332f7af-f7d6-49db-8d6f916ab8b7ab44.pdf&p=AMDT"">AMDT",{ts '2026-10-01 00:00:00'},AMDT 40-26
6,2322,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=9705553d-7f0e-409a-8037481337de8e5f.pdf&p=AMDT"">AMDT",{ts '2026-10-01 00:00:00'},AMDT 40-26
7,2322,AIP,AMDT,"https://aisweb.decea.gov.br/download/?public=ab511a27-f41b-49be-a8b65084bd9ff67a.pdf&p=AMDT"">AMDT",{ts '2026-10-01 00:00:00'},AMDT 40-26
8,2322,AIP,Completa,"https://aisweb.decea.gov.br/download/?public=c234cede-a1ad-4776-85a338ca53bd5240.pdf&p=Completa"">Completa",{ts '2026-10-01 00:00:00'},AMDT 40-26
9,2322,AIP,Completa,"https://aisweb.decea.gov.br/download/?public=be5d1dea-d34e-4eb5-b7ebf0d19811cb09.pdf&p=Completa"">Completa",{ts '2026-10-01 00:00:00'},AMDT 40-26


## 11. `waypoints`: fixos de navegação

~6,8 mil pontos em páginas de 100, ou seja, cerca de 70 requisições (leva uns 30 s). Use `max_paginas` para testar com menos.

In [25]:
df_wpt = para_df(listar_paginado("waypoints"))
df_wpt["latitude"] = df_wpt["latitude"].astype(float)
df_wpt["longitude"] = df_wpt["longitude"].astype(float)
df_wpt.describe()

waypoints: 6785 itens em 68 página(s), total informado = 6785


,latitude,longitude
count,6785.000000,6785.000000
mean,-17.516663,-48.360520
std,8.822346,7.645977
min,-33.224500,-73.678333
25%,-23.391136,-51.847944
50%,-20.680000,-48.074778
75%,-10.380833,-43.385244
max,8.462833,-10.000000


## Próximos passos

Algumas ideias para o trabalho a partir daqui:
- Cruzar `rotaer` (lat/lon) com `notam` para ver quais regiões concentram mais restrições.
- Coletar `met` periodicamente para montar uma série histórica de METARs de alguns aeródromos (a API só devolve o mais recente).
- Plotar `waypoints` e aeródromos num mapa (ex.: `folium` ou `plotly`).